# ABSA Evaluation Notebook
Dieses Notebook evaluiert die Ergebnisse der Fine-Tuning Experimente gegen die Gold-Standard Daten unter Verwendung des ABSA Toolkits.

In [1]:
import sys
import os
import json

TOOLKIT_PATH = '/home/hellwig/absa-toolkit'
sys.path.append(TOOLKIT_PATH)

from helper import get_dataset, get_all_scores
from paraphrase import compute_f1_scores

/home/hellwig/miniconda3/envs/vllm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Parameter für den Datensatz und die JSON-Datei
dataset_name = "rest16"
task = "asqp"
seed_run = 0

# Gold-Daten laden (wie in test_gemma4.py)
test_data_raw = get_dataset(dataset_name, "test", task, TOOLKIT_PATH + "/data")
gold_labels = [[list(tupl) for tupl in example["label"]] for example in test_data_raw]

print(f"Lade {len(gold_labels)} Gold-Beispiele für {dataset_name} ({task})")

Lade 544 Gold-Beispiele für rest16 (asqp)


In [3]:
# Vorhersagedaten laden
path_results = f"fine_tuning_results_gemma_4/results_llm_{dataset_name}_{task}_{seed_run}.json"

with open(path_results, "r") as f:
    results = json.load(f)

pred_labels = results["all_preds"]
print(f"Geladen: {len(pred_labels)} Vorhersagen aus {path_results}")

Geladen: 544 Vorhersagen aus fine_tuning_results_gemma_4/results_llm_rest16_asqp_0.json


In [4]:
# Evaluierung mit get_all_scores ausführen
# Hinweis: get_all_scores erwartet Listen von Listen von Listen (oder ähnliche Strukturen)
scores = get_all_scores(pred_labels, gold_labels)

# Ergebnisse anzeigen
print("\n--- Evaluierungsergebnisse ---")
for key, value in scores.items():
    print(f"{key}: {value:.4f}")

# Optional: Vergleich mit compute_f1_scores (falls get_all_scores nicht alles abdeckt)
f1_results = compute_f1_scores(pred_labels, gold_labels)
print("\n--- F1 Scores Details ---")
print(json.dumps(f1_results, indent=2))


--- Evaluierungsergebnisse ---
precision: 64.7199
recall: 67.9599
f1: 66.3004
TP: 543.0000
FP: 296.0000
FN: 256.0000
f1_macro: 58.7978
number of gold spans: 799, predicted spans: 839, hit: 543

--- F1 Scores Details ---
{
  "precision": 0.6471990464839095,
  "recall": 0.6795994993742178,
  "f1": 0.6630036630036631
}
